# Build Unlabeled Pretraining Dataset

Create a broad unlabeled tensor dataset once so CNN and transformer pretraining notebooks can reuse the same artifact.

In [1]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

# The full 2200-instance 96x96 tensor cache is about 7.6 GiB. Keep enough
# cache budget for resume/restart behavior while still leaving disk headroom.
os.environ.setdefault("ZF_TENSOR_CACHE_MAX_BYTES", "12G")
os.environ.setdefault("ZF_DATASET_CACHE_MAX_BYTES", "12G")
os.environ.setdefault("ZF_CACHE_MIN_FREE_BYTES", "2G")

from src.tensor_utils import build_unlabeled_tensor_dataset, load_unlabeled_tensor_dataset
from src.notebook_utils import configure_full_dataframe_display, load_compound_image_condition_map_csv

configure_full_dataframe_display()

In [2]:
# User inputs

unlabeled_dataset_dir = Path(".dataset_cache/unlabeled_active_high_mid_low_t20_z5_y96_x96_chunks")
chunk_size = 100
rebuild_unlabeled_dataset = False

selected_mechanisms = None
selected_concentrations = ["high", "mid", "low"]
include_treatments = True
include_controls = True
max_tensors_per_compound = None
max_tensors_total = None

output_size = (20, 5, 96, 96)
only_active = True
normalize_global_drift = True
loess_frac = 0.25
use_cache = True
use_tiff_cache = True

In [ ]:
if rebuild_unlabeled_dataset or not (unlabeled_dataset_dir / "manifest.json").exists():
    condition_df = load_compound_image_condition_map_csv()
    unlabeled_dataset = build_unlabeled_tensor_dataset(
        condition_df=condition_df,
        output_size=output_size,
        selected_mechanisms=selected_mechanisms,
        selected_concentrations=selected_concentrations,
        include_treatments=include_treatments,
        include_controls=include_controls,
        max_tensors_per_compound=max_tensors_per_compound,
        max_tensors_total=max_tensors_total,
        only_active=only_active,
        normalize_global_drift=normalize_global_drift,
        loess_frac=loess_frac,
        use_cache=use_cache,
        use_tiff_cache=use_tiff_cache,
        chunk_output_dir=unlabeled_dataset_dir,
        chunk_size=chunk_size,
        overwrite_chunks=rebuild_unlabeled_dataset,
    )

unlabeled_dataset = load_unlabeled_tensor_dataset(unlabeled_dataset_dir)

unlabeled_dataset["tensors"].shape, unlabeled_dataset["metadata"].shape, unlabeled_dataset_dir

[2026-05-07 15:05:59] [001/2200] kind=control   source=tensor_cache conc=control  elapsed=00:01 eta=32:29 mechanism=AChE_Inhibitor_Reversible compound=Donepezil
[2026-05-07 15:06:00] [002/2200] kind=control   source=tensor_cache conc=control  elapsed=00:03 eta=47:52 mechanism=AChE_Inhibitor_Reversible compound=Donepezil
[2026-05-07 15:06:01] [003/2200] kind=control   source=tensor_cache conc=control  elapsed=00:03 eta=35:33 mechanism=AChE_Inhibitor_Reversible compound=Donepezil
[2026-05-07 15:06:01] [004/2200] kind=control   source=tensor_cache conc=control  elapsed=00:04 eta=33:34 mechanism=AChE_Inhibitor_Reversible compound=Donepezil
[2026-05-07 15:06:02] [005/2200] kind=control   source=tensor_cache conc=control  elapsed=00:04 eta=29:16 mechanism=AChE_Inhibitor_Reversible compound=Donepezil
[2026-05-07 15:06:02] [006/2200] kind=control   source=tensor_cache conc=control  elapsed=00:04 eta=27:10 mechanism=AChE_Inhibitor_Reversible compound=Donepezil
[2026-05-07 15:06:03] [007/2200] k

In [ ]:
display(
    unlabeled_dataset["metadata"][["mechanism_of_action", "compound", "condition_kind", "concentration_band"]]
    .value_counts()
    .rename("n_samples")
    .reset_index()
    .head(40)
)